<!-- This is the beginning code to clean ebird for LA  -->

# **Part 1.** Cleaning Ebird Data 

Updated: 11.08.2026 
Author: Oliver Nguyen


This code borrows from eBird best practices and Callaghan et al. 2021 to work with citizen science data, we'll be using auk package to help us clean the data and using R. We downloaded respective datasets from San Francisco and Los Angeles. 


The main filters we will be using are: 

[x] Years 2010 - 2025 (Jan - Dec)  
[x] => 0.5 km distance  
[x] area <50 ha
[x] => 10 observers  
[x] combine shared checklists (Auk_Unique)  
[x] => 15- 120 minutes hours total efforts checklist  
[x] Stationary + Travelling + Area checklists  
[x] Completed only checklists  

We'll be aiming to be using **Incidence Frequency** instead of Abundance (Callaghan et al. 2022) as it would be more stable. 

According to Auk_ebd, the filters we cna use are: 

#> Filters 
#>   Species: all
#>   Countries: USA
#>   States: California
#>   Counties: SF County/LA County
#>   BCRs: NA
#>   Bounding box: full extent
#>   Years: 2015 - 2026
#>   Date: all
#>   Start time: NA
#>   Last edited date: NA
#>   Protocol: Travelling, Stationary, Area
#>   Project code: NA
#>   Duration: 15 - 120 minutes
#>   Distance travelled: => 1 km 
#>   Records with breeding codes only: no
#>   Exotic Codes: all
#>   Complete checklists only: yes


In [1]:

#Libraries needed
import pandas as pd
import geopandas as gpd
from datetime import datetime
from pathlib import Path
import numpy as np
import duckdb as duckdb

# Import rpy2 components
#%conda install rpy2

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, Formula
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter


## Steps in Filtering


1. Import checklists and observations
2. Save to Parquets and process in chunks 
    * Convert and process effort hours 
    * auk roll-up 
    * zero-filling
4. Filter efforts (observation hours, distance, number of observers)
5. Seperate into stationary and travelling checklists
3. Keep only max observations of species per day 



In [3]:
%conda install -c conda-forge r-arrow #If you need to install R packages, you ned ot run conda 

Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/oliver/miniforge3/envs/sf_env

  added / updated specs:
    - r-arrow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    r-arrow-23.0.1             |    r44hecca717_0         4.4 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         4.4 MB

The following NEW packages will be INSTALLED:

  r-arrow            conda-forge/linux-64::r-arrow-23.0.1-r44hecca717_0 



                                                                                
Preparing transaction: done
Verifying transaction: done
Executing transaction: done
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get s

In [4]:

# ============================================================================
# Import R packages
# ============================================================================

print("Loading R packages...")

# Import R packages
base = importr('base')
utils = importr('utils')
auk = importr('auk')
lubridate = importr('lubridate')
dplyr = importr('dplyr')
sf = importr('sf')
arrow = importr('arrow')

# Resolve namespace conflicts in R
ro.r('select <- dplyr::select')
ro.r('filter <- dplyr::filter')
print("R packages loaded successfully!")


Loading R packages...
R packages loaded successfully!


In [5]:

# ============================================================================
# eBird Data Processing 
# ============================================================================

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from pathlib import Path
import pandas as pd



In [6]:

# ============================================================================
# 3. FAST PROCESSING WITH auk PACKAGE
# ============================================================================

ro.r(r'''
    library(auk)
    library(dplyr)
    library(lubridate)
    library(sf)
    library(data.table)
    
    # Resolve namespace conflicts
    select <- dplyr::select
    filter <- dplyr::filter
    
    n_cores <- parallel::detectCores() - 1
    setDTthreads(n_cores)
    
    cat("Using", n_cores, "cores\n")
    

''')


R callback write-console: data.table 1.18.2.1 using 4 threads (see ?getDTthreads).    
R callback write-console: Latest news: r-datatable.com
  
R callback write-console: 
Attaching package: ‘data.table’

  
R callback write-console: The following objects are masked from ‘package:dplyr’:

    between, first, last

  
R callback write-console: The following objects are masked from ‘package:lubridate’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year

  


Using 7 cores


In [ ]:
#Takes about 8 minutes 

ro.r(r'''
library(auk)
library(dplyr)
library(lubridate)
library(arrow)

#Import in the sampling and ebird data you got
#Reccomended to do this locally first
f_ebd_input <- "/home/oliver/Documents/ebd_US-CA-075_201001_202612_smp_relSep-2026/ebd_US-CA-075_201001_202612_smp_relSep-2026.txt"
f_sed_input <- "/home/oliver/Documents/ebd_US-CA-075_201001_202612_smp_relSep-2026/ebd_US-CA-075_201001_202612_smp_relSep-2026_sampling.txt"
out_dir     <- "/home/oliver/Documents/data/output"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# New file names so the old stationary-only files aren't reused
f_ebd <- file.path(out_dir, "ebd_SF_stat_trav_area.txt")
f_sed <- file.path(out_dir, "sed_SF_stat_trav_area.txt")
zf_dir <- file.path(out_dir, "zf_stat_trav_area")

# ---- ONE pass through the raw files (filters auk can do) ----
if (!file.exists(f_ebd)) {
  auk_ebd(f_ebd_input, file_sampling = f_sed_input) |>
    auk_date(c("2010-01-01", "2025-12-31")) |>                  # matches the file and the loop years
    auk_protocol(c("Stationary", "Traveling", "Area")) |>        # eBird spelling: "Traveling"
    auk_duration(c(15, 120)) |>                                  # 15-120 minutes
    auk_complete() |>                                            # complete checklists only
    auk_filter(file = f_ebd, file_sampling = f_sed, overwrite = TRUE)
}

# ---- Read (unique = TRUE by default collapses shared/group checklists = auk_unique) ----
sed <- read_sampling(f_sed, unique = TRUE) |>
  filter(
    number_observers <= 10,
    case_when(
      protocol_name == "Stationary" ~ TRUE,
      protocol_name == "Traveling"  ~ !is.na(effort_distance_km) & effort_distance_km <= .5,
      protocol_name == "Area"       ~ !is.na(effort_area_ha)     & effort_area_ha < 50,
      TRUE ~ FALSE
    )
  )

ebd <- read_ebd(f_ebd, unique = TRUE) |>
  filter(checklist_id %in% sed$checklist_id)

# Check what survived each protocol
print(count(sed, protocol_name))
print(sed |> group_by(protocol_name) |>
        summarise(med_min  = median(duration_minutes),
                  med_km   = median(effort_distance_km, na.rm = TRUE),
                  med_ha   = median(effort_area_ha, na.rm = TRUE)))

time_to_decimal <- function(x) {
  x <- hms(x, quiet = TRUE); hour(x) + minute(x)/60 + second(x)/3600
}

# ---- Zero-fill per year, write Parquet ----
for (yr in 2010:2025) {
  sed_y <- filter(sed, year(observation_date) == yr)
  if (nrow(sed_y) == 0) next
  ebd_y <- filter(ebd, checklist_id %in% sed_y$checklist_id)

  zf <- auk_zerofill(ebd_y, sed_y, collapse = TRUE) |>
    mutate(
      observation_count  = if_else(species_observed,
                                   suppressWarnings(as.integer(observation_count)), 0L),  # "X" -> NA
      # keep real distances; stationary = 0, area = NA (area is in effort_area_ha)
      effort_distance_km = if_else(protocol_name == "Stationary", 0, effort_distance_km),
      effort_hours       = duration_minutes / 60,
      effort_speed_kmph  = effort_distance_km / effort_hours,
      hours_of_day       = time_to_decimal(time_observations_started),
      year               = year(observation_date),
      month              = month(observation_date),
      day_of_year        = yday(observation_date)
    )

  write_dataset(zf, zf_dir, format = "parquet",
                partitioning = c("year", "month"),
                existing_data_behavior = "delete_matching")
  cat(sprintf("%d: %s rows\n", yr, format(nrow(zf), big.mark = ",")))
  rm(zf, ebd_y, sed_y); gc()
}
''')



# A tibble: 3 × 2
  protocol_name     n
  <chr>         <int>
1 Area           3310
2 Stationary    19134
3 Traveling     28516
# A tibble: 3 × 4
  protocol_name med_min med_km med_ha
  <chr>           <dbl>  <dbl>  <dbl>
1 Area               48  0.267   2.02
2 Stationary         30  0.161   1.82
3 Traveling          35  0.322   1.62
2010: 124,080 rows
2011: 284,756 rows
2012: 463,904 rows
2013: 601,481 rows
2014: 602,184 rows
2015: 614,368 rows
2016: 611,754 rows
2017: 609,604 rows
2018: 839,120 rows
2019: 960,868 rows
2020: 1,229,937 rows
2021: 1,415,595 rows
2022: 1,274,076 rows
2023: 1,304,980 rows
2024: 1,431,648 rows
2025: 1,591,140 rows


R callback write-console: In addition:   
R callback write-console: There were 16 warnings (use warnings() to see them)  
R callback write-console: 
  


In [ ]:
import duckdb

#be careful to go over the RAM limit

zf_dir  = "/home/oliver/Documents/data/output/zf_stat_trav_area"
out_pq  = "/home/oliver/Documents/data/output/zf_SF_stat_trav_area_2010_2025.parquet"

con = duckdb.connect()
con.execute(f"""
  COPY (
    SELECT *
    FROM read_parquet('{zf_dir}/**/*.parquet', hive_partitioning = true)
    ORDER BY observation_date
  ) TO '{out_pq}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Check: rows and checklists per year
print(con.execute(f"""
  SELECT year, COUNT(*) AS n_rows, COUNT(DISTINCT checklist_id) AS n_checklists
  FROM read_parquet('{out_pq}')
  GROUP BY year ORDER BY year
""").df())
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    year   n_rows  n_checklists
0   2010   124080           564
1   2011   284756          1108
2   2012   463904          1744
3   2013   601481          2287
4   2014   602184          2281
5   2015   614368          2336
6   2016   611754          2418
7   2017   609604          2372
8   2018   839120          3085
9   2019   960868          3572
10  2020  1229937          4377
11  2021  1415595          4967
12  2022  1274076          4518
13  2023  1304980          4595
14  2024  1431648          4971
15  2025  1591140          5765


In [ ]:
# This takes about 103 mins
# Remote into the HPC or use Sherlock
# ============================================================================
# 1. Process eBird data in monthly chunks with stationary filters
# ============================================================================

ro.r(r'''
library(auk)
library(dplyr)
library(lubridate)
library(sf)
library(janitor)  # For clean_names()

# Resolve namespace conflicts
select <- dplyr::select
filter <- dplyr::filter

# Set parameters
rerun <- TRUE  # Set to TRUE to rerun filtering
start_year <- 2010
end_year <- 2025
end_month <- 12

# Set input file paths (your original eBird files)
f_ebd_input <- "D:/Chapter1_SF_redlining_Rprj/ebd_US-CA-075_201001_202512_smp_relMar-2026/ebd_US-CA-075_201001_202512_smp_relMar-2026.txt"
f_sed_input <- "D:/Chapter1_SF_redlining_Rprj/ebd_US-CA-075_201001_202512_smp_relMar-2026/ebd_US-CA-075_201001_202512_smp_relMar-2026_sampling."



# Create output directory if it doesn't exist
dir.create("D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked", showWarnings = FALSE, recursive = TRUE)

# ========================================================================
# STEP A: Filter data into monthly chunks
# ========================================================================

if (rerun) {
  cat("\n================================================================================\n")
  cat("FILTERING DATA INTO MONTHLY CHUNKS (STATIONARY ONLY)\n")
  cat("================================================================================\n")
  
  list_years <- seq(start_year, end_year)
  total_months <- 0
  
  # Count total months to process
  for (list_year in list_years) {
    for (list_month in seq(1, 12)) {
      if (list_year < end_year | (list_year == end_year & list_month <= end_month)) {
        total_months <- total_months + 1
      }
    }
  }
  
  cat(sprintf("\nProcessing %d months from %d-%02d to %d-%02d\n", 
              total_months, start_year, 1, end_year, end_month))
  
  current_month <- 0
  
  # Function to convert time to decimal hours (define once outside loop)
  time_to_decimal <- function(x) {
    x <- hms(x, quiet = TRUE)
    hour(x) + minute(x) / 60 + second(x) / 3600
  }
  
  for (list_year in list_years) {
    for (list_month in seq(1, 12)) {
      if (list_year < end_year | (list_year == end_year & list_month <= end_month)) {
        
        current_month <- current_month + 1
        
        # Create date range to use in auk filter
        start_date <- make_date(year = list_year, month = list_month, day = 1L)
        end_date <- ceiling_date(start_date, unit = "month") - 1
        
        cat(sprintf("\n[%d/%d] Processing %s to %s...\n", 
                    current_month, total_months, start_date, end_date))
        
        # Create output filenames based on month and year
        file_ebd <- file.path('D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked', 
                              paste0('ebd_filtered_SF_', list_year, '_', 
                                     sprintf("%02d", list_month), '.txt'))
        file_sampling <- file.path('D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked', 
                                   paste0('sampling_filtered_SF_', list_year, '_', 
                                          sprintf("%02d", list_month), '.txt'))
        
        # Filter data using auk
        cat("  Filtering data...\n")
        auk_ebd(f_ebd_input, file_sampling = f_sed_input) %>%
          auk_date(date = c(start_date, end_date)) %>%
          auk_complete() %>%  # Only complete checklists
          auk_filter(file = file_ebd, 
                     file_sampling = file_sampling,
                     overwrite = TRUE)
     
        # ======================================================
        # STEP B: Read the filtered data with zero-fill
        # ======================================================
        cat("  Reading filtered data...\n")
        ebd_zf <- auk_zerofill(x = file_ebd, 
                               sampling_events = file_sampling, 
                               collapse = TRUE)
        
        cat(sprintf("    Observations before filtering: %s\n", 
                    format(nrow(ebd_zf), big.mark = ",")))
        
        # Standardize column names to lowercase with underscores
        cat("  Standardizing column names...\n")
        ebd_zf <- ebd_zf %>%
          clean_names(case = "snake")
        
        cat(sprintf("    Column names cleaned (first 5): %s\n", 
                    paste(head(names(ebd_zf), 5), collapse = ", ")))
     
        # ====================================
        # STEP C: Clean up variables, process, and FILTER
        # ====================================
        cat("  Processing variables and applying filters...\n")
        
        zf <- ebd_zf %>%
          mutate(
            # Calculate derived variables first
            observation_count = if_else(species_observed, 
                                       as.integer(observation_count), 
                                       0L),
            effort_distance_km = if_else(protocol_name == "Stationary", 
                                        0, effort_distance_km),
            effort_hours = duration_minutes / 60,
            effort_speed_kmph = if_else(effort_hours > 0,
                                       effort_distance_km / effort_hours,
                                       0),
            hours_of_day = time_to_decimal(time_observations_started),
            year = year(observation_date),
            day_of_year = yday(observation_date)
          ) %>%

          # Apply filters for stationary protocols with effort constraints
          filter(
            protocol_name == "Stationary",
            effort_hours <= 2, #keep to two hours or less
            effort_distance_km <= 1, #change effort distance to less than 1 km
            effort_speed_kmph <= 5,
            number_observers <= 10,
            
          )
          %>%
          
        # Shared checklists (same birding party) count as one sampling event
        mutate(event = coalesce(group_id, checklist_id)) %>%
        
        cat(sprintf("    Observations after filtering: %s\n", 
                    format(nrow(zf), big.mark = ",")))
        
        # Show filtering statistics
        if (nrow(ebd_zf) > 0) {
          pct_kept <- 100 * nrow(zf) / nrow(ebd_zf)
          cat(sprintf("    Kept: %.1f%% (removed %s observations)\n", 
                      pct_kept,
                      format(nrow(ebd_zf) - nrow(zf), big.mark = ",")))
        }
        
        # Only save if there's data
        if (nrow(zf) > 0) {
          # Save the processed data
          output_file <- file.path('D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked', 
                                  paste0('ebd_processed_stationary_SF_', list_year, '_', 
                                         sprintf("%02d", list_month), '.csv'))
          write.csv(zf, output_file, row.names = FALSE)
          
          cat(sprintf("  ✓ Processed and saved: %s\n", basename(output_file)))
        } else {
          cat("  ⚠ No observations passed filters, skipping save.\n")
        }
        
        # Clean up memory
        rm(ebd_zf, zf)
        gc(verbose = FALSE)
      }
    }
  }
  
  cat("\n✓ All monthly chunks filtered and processed successfully!\n")
  
  # ========================================================================
  # Summary of all processed files
  # ========================================================================
  cat("\n================================================================================\n")
  cat("SUMMARY\n")
  cat("================================================================================\n")
  
  processed_files <- list.files("D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked",
                                pattern = "^ebd_processed_stationary_SF_.*\\.csv$",
                                full.names = FALSE)
  
  cat(sprintf("\nTotal monthly files created: %d\n", length(processed_files)))
  cat("\nFilters applied:\n")
  cat("  - Protocol: Stationary only\n")
  cat("  - Effort hours: <= 2\n")
  cat("  - Effort distance: <= 1 km\n")
  cat("  - Effort speed: <= 5 km/h\n")
  cat("  - Number of observers: <= 10\n")
}
''')



FILTERING DATA INTO MONTHLY CHUNKS (STATIONARY ONLY)

Processing 192 months from 2010-01 to 2025-12

[1/192] Processing 2010-01-01 to 2010-01-31...
  Filtering data...
  Reading filtered data...
    Observations before filtering: 28,500
  Standardizing column names...
    Column names cleaned (first 5): checklist_id, last_edited_date, country, country_code, state
  Processing variables and applying filters...
    Observations after filtering: 3,750
    Kept: 13.2% (removed 24,750 observations)
  <U+2713> Processed and saved: ebd_processed_stationary_SF_2010_01.csv

[2/192] Processing 2010-02-01 to 2010-02-28...
  Filtering data...
  Reading filtered data...
    Observations before filtering: 19,728
  Standardizing column names...
    Column names cleaned (first 5): checklist_id, last_edited_date, country, country_code, state
  Processing variables and applying filters...
    Observations after filtering: 1,781
    Kept: 9.0% (removed 17,947 observations)
  <U+2713> Processed and saved

R callback write-console: In addition:   
R callback write-console: There were 50 or more warnings (use warnings() to see the first 50)  
R callback write-console: 
  


In [ ]:
# =======================================
# 2. Combine monthly files, add max observations, train/test split
# =======================================

import duckdb
import pandas as pd
import os

print("\n" + "="*80)
print("STEP 2: COMBINING MONTHLY FILES AND ADDING MAX OBSERVATIONS")
print("="*80 + "\n")

# Create a specific temp directory on D drive
temp_dir = "D:/Chapter1_LA_redlining/data/temp"
os.makedirs(temp_dir, exist_ok=True)

# Connect to DuckDB with temp directory configuration
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
con.execute("SET threads = 1")
con.execute("PRAGMA max_temp_directory_size='300GiB'")
con.execute(f"SET temp_directory='{temp_dir}'")

# Get all monthly CSV files
monthly_pattern = "D:/Chapter1_SF_redlining_Rprj/data/ebd_US-SF_chunked/ebd_processed_stationary_SF_*.csv"

print("Reading monthly stationary files...")

# Get distinct years from the monthly files
years = con.execute(f"""
  SELECT DISTINCT YEAR(observation_date) as year 
  FROM read_csv_auto('{monthly_pattern}', 
                     union_by_name=true, 
                     ignore_errors=true,
                     nullstr='NA')
  ORDER BY year
""").fetchall()

years = [y[0] for y in years]
print(f"Processing years: {years}\n")

# ========================================================================
# Step 2a: Create train/test split
# ========================================================================
print("--- Creating train/test split ---")

first_year = years[0]
print(f"Creating table with year {first_year}...")
con.execute(f"""
  CREATE TABLE observations_split AS
  SELECT *, 
         CASE WHEN random() <= 0.8 THEN 'train' ELSE 'test' END as type
  FROM read_csv_auto('{monthly_pattern}', 
                     union_by_name=true, 
                     ignore_errors=true,
                     nullstr='NA')
  WHERE YEAR(observation_date) = {first_year}
""")

# Append remaining years
for year in years[1:]:
    print(f"Processing year {year}...")
    con.execute(f"""
      INSERT INTO observations_split
      SELECT *, 
             CASE WHEN random() <= 0.8 THEN 'train' ELSE 'test' END as type
      FROM read_csv_auto('{monthly_pattern}', 
                         union_by_name=true, 
                         ignore_errors=true,
                         nullstr='NA')
      WHERE YEAR(observation_date) = {year}
    """)

print("✓ Train/test split complete!\n")

# ========================================================================
# Step 2b: Add max_observations per species per day
# ========================================================================
print("--- Adding max_observations column ---")

years = con.execute("""
  SELECT DISTINCT YEAR(observation_date) as year 
  FROM observations_split 
  ORDER BY year
""").fetchall()
years = [y[0] for y in years]

first_year = years[0]
print(f"Processing year {first_year}...")
con.execute(f"""
  CREATE TABLE observations_with_max AS
  SELECT *,
         MAX(COALESCE(TRY_CAST(observation_count AS INTEGER), 0)) OVER (
           PARTITION BY scientific_name, observation_date
         ) as max_observations
  FROM observations_split
  WHERE YEAR(observation_date) = {first_year}
""")

for year in years[1:]:
    print(f"Processing year {year}...")
    con.execute(f"""
      INSERT INTO observations_with_max
      SELECT *,
             MAX(COALESCE(TRY_CAST(observation_count AS INTEGER), 0)) OVER (
               PARTITION BY scientific_name, observation_date
             ) as max_observations
      FROM observations_split
      WHERE YEAR(observation_date) = {year}
    """)

print("✓ Max observations calculated!\n")

# ========================================================================
# Step 2c: Keep only max observations (filter out non-max)
# ========================================================================
print("--- Keeping only maximum observations ---")

# Process in smaller chunks to avoid temp space issues
years = con.execute("""
  SELECT DISTINCT YEAR(observation_date) as year 
  FROM observations_with_max 
  ORDER BY year
""").fetchall()
years = [y[0] for y in years]

first_year = years[0]
print(f"Filtering year {first_year}...")
con.execute(f"""
  CREATE TABLE observations_max_only AS
  SELECT *
  FROM observations_with_max
  WHERE YEAR(observation_date) = {first_year}
    AND COALESCE(TRY_CAST(observation_count AS INTEGER), 0) = max_observations
""")

for year in years[1:]:
    print(f"Filtering year {year}...")
    con.execute(f"""
      INSERT INTO observations_max_only
      SELECT *
      FROM observations_with_max
      WHERE YEAR(observation_date) = {year}
        AND COALESCE(TRY_CAST(observation_count AS INTEGER), 0) = max_observations
    """)

print("✓ Filtered to max observations only!\n")

# Check results
result = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        MIN(observation_date) as min_date,
        MAX(observation_date) as max_date,
        COUNT(DISTINCT scientific_name) as n_species,
        COUNT(DISTINCT sampling_event_identifier) as n_checklists
    FROM observations_max_only
""").df()

print("\nAfter keeping only max observations:")
print(result)

# Train/test breakdown
split_summary = con.execute("""
    SELECT 
        type,
        COUNT(*) as n_observations,
        COUNT(DISTINCT sampling_event_identifier) as n_checklists,
        COUNT(DISTINCT scientific_name) as n_species,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as pct
    FROM observations_max_only
    GROUP BY type
""").df()
print("\nTrain/Test Split:")
print(split_summary)

# Check data quality
print("\nData Quality Check:")
quality_check = con.execute("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN observation_count IS NULL THEN 1 ELSE 0 END) as null_counts,
        SUM(CASE WHEN max_observations = 0 THEN 1 ELSE 0 END) as zero_max,
        MIN(max_observations) as min_max,
        MAX(max_observations) as max_max,
        AVG(max_observations) as avg_max
    FROM observations_max_only
""").df()
print(quality_check)

# Save intermediate result
print("\nSaving intermediate file (before spatial filter)...")
con.execute("""
  COPY observations_max_only 
  TO 'D:/Chapter1_SF_redlining_Rprj/data/output/observations_max_prespatial.parquet'
  (FORMAT PARQUET)
""")
print("✓ Saved: observations_max_prespatial.parquet")

print("\n" + "="*80)
print("STEP 2 COMPLETE!")
print("="*80 + "\n")

con.close()

# Clean up temp directory if desired
# import shutil
# shutil.rmtree(temp_dir, ignore_errors=True)


STEP 2: COMBINING MONTHLY FILES AND ADDING MAX OBSERVATIONS

Reading monthly stationary files...
Processing years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

--- Creating train/test split ---
Creating table with year 2010...
Processing year 2011...
Processing year 2012...
Processing year 2013...
Processing year 2014...
Processing year 2015...
Processing year 2016...
Processing year 2017...
Processing year 2018...
Processing year 2019...
Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...
Processing year 2025...
✓ Train/test split complete!

--- Adding max_observations column ---
Processing year 2010...
Processing year 2011...
Processing year 2012...
Processing year 2013...
Processing year 2014...
Processing year 2015...
Processing year 2016...
Processing year 2017...
Processing year 2018...
Processing year 2019...
Processing year 2020...
Processing year 2021...
Pr

IOException: IO Error: Cannot open file "D:/Chapter1_SF_redlining_Rprj/data/output/observations_max_prespatial.parquet": The system cannot find the path specified.


In [8]:
# Save intermediate result
print("\nSaving intermediate file (before spatial filter)...")
con.execute("""
  COPY observations_max_only 
  TO 'D:/Chapter1_SF_redlining_Rprj/data/output/observations_max_prespatial.parquet'
  (FORMAT PARQUET)
""")
print("✓ Saved: observations_max_prespatial.parquet")

print("\n" + "="*80)
print("STEP 2 COMPLETE!")
print("="*80 + "\n")

con.close()


Saving intermediate file (before spatial filter)...
✓ Saved: observations_max_prespatial.parquet

STEP 2 COMPLETE!



In [ ]:

# Save intermediate result
print("\nSaving intermediate file (before spatial filter)...")
con.execute("""
  COPY observations_max_only 
  TO 'D:/Chapter1_SF_redlining_Rprj/data/output/observations_max_prespatial.parquet'
  (FORMAT PARQUET)
""")
print("✓ Saved: observations_max_prespatial.parquet")

print("\n" + "="*80)
print("STEP 2 COMPLETE!")
print("="*80 + "\n")

con.close()

# Clean up temp directory if desired
# import shutil
# shutil.rmtree(temp_dir, ignore_errors=True)

Stop here and move to Part 2: LA_holc_join_12.13.25.ipynb 

We should have something to join to the HOLC now 


In [ ]:
#This part is to briefly look at our data sets
#free up some space 

ro.r(f'''
breaks <- seq(0, 24)
labels <- breaks[-length(breaks)] + diff(breaks) / 2
checklists_time <- checklists |> 
  mutate(hour_bins = cut(hours_of_day, 
                         breaks = breaks, 
                         labels = labels,
                         include.lowest = TRUE),
         hour_bins = as.numeric(as.character(hour_bins))) |> 
  group_by(hour_bins) |> 
  summarise(n_checklists = n(),
            n_detected = sum(species_observed),
            det_freq = mean(species_observed))
''')

In [ ]:

ro.r(r'''
# Get top 20 species by total observation count
top_species <- zf_filtered %>%
  group_by(scientific_name) %>%
  summarize(total_count = sum(observation_count), na.rm = TRUE) %>%
  arrange(desc(total_count)) %>%
  head(20) %>%
  pull(scientific_name)

top_species


# Plot only those top species
zf_filtered %>%
  filter(scientific_name %in% top_species) %>%
  ggplot(aes(x = observation_date, y = scientific_name)) +
  geom_tile() +
  scale_fill_viridis_c(name = "Year", option = "plasma") +
  labs(
    title = "Top 20 Bird Species Observations Over Time",
    x = "Date",
    y = "Species",
    fill = "Observation Count"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text.y = element_text(size = 10)
  )

#Get species sighted per day

top_species <- zf_filtered %>%
  group_by(observation_date) %>%
  summarize(total_count = count(unique(scientific_name)), na.rm = TRUE) %>%
  arrange(desc(total_count)) %>%
  head(20) %>%
  pull(scientific_name)

    '''
)



#Now let's look at Checklist variation 
ro.r(f'''
# summarize data by hour long bins
breaks <- seq(0, 6)
labels <- breaks[-length(breaks)] + diff(breaks) / 2
checklists_duration <- checklists |> 
  mutate(duration_bins = cut(effort_hours, 
                             breaks = breaks, 
                             labels = labels,
                             include.lowest = TRUE),
         duration_bins = as.numeric(as.character(duration_bins))) |> 
  group_by(duration_bins) |> 
  summarise(n_checklists = n(),
            n_detected = sum(species_observed),
            det_freq = mean(species_observed))

# histogram
g_duration_hist <- ggplot(checklists_duration) +
  aes(x = duration_bins, y = n_checklists) +
  geom_segment(aes(xend = duration_bins, y = 0, yend = n_checklists),
               color = "grey50") +
  geom_point() +
  scale_x_continuous(breaks = breaks) +
  scale_y_continuous(labels = scales::comma) +
  labs(x = "Checklist duration [hours]",
       y = "# checklists",
       title = "Distribution of checklist durations")

# frequency of detection
g_duration_freq <- ggplot(checklists_duration |> filter(n_checklists > 100)) +
  aes(x = duration_bins, y = det_freq) +
  geom_line() +
  geom_point() +
  scale_x_continuous(breaks = breaks) +
  scale_y_continuous(labels = scales::percent) +
  labs(x = "Checklist duration [hours]",
       y = "% checklists with detections",
       title = "Detection frequency")

# combine
grid.arrange(g_duration_hist, g_duration_freq)

''')

